In [35]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torchvision.transforms as T
import timm
from transformers import AutoModel
from sklearn.cluster import DBSCAN
from wildlife_datasets.datasets import AnimalCLEF2026
from wildlife_tools.features import DeepFeatures
from wildlife_tools.similarity import CosineSimilarity
from wildlife_datasets import analysis, datasets, loader
from dotenv import load_dotenv
load_dotenv() 

root = 'data'
dataset_full = AnimalCLEF2026(
    root,
    transform=None,
    load_label=True,
    factorize_label=True,
    check_files=False
)

train_transform = T.Compose([
    T.Resize([384, 384]),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
]) 

test_transform = T.Compose([
    T.Resize([384, 384]),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
]) 

# print(dataset_full.metadata.head())
# print(dataset_full.metadata[['dataset', 'split']].value_counts(sort=False))

dataset_train = dataset_full.get_subset(dataset_full.df['split'] == 'train')
dataset_train.set_transform(train_transform)  
dataset_test = dataset_full.get_subset(dataset_full.df['split'] == 'test')
dataset_test.set_transform(test_transform)
# datasets = {}
# for name in dataset_train.metadata['dataset'].unique():
#     datasets[name + "_train"] = dataset_train.get_subset(dataset_train.df['dataset'] == name)

# for name in dataset_test.metadata['dataset'].unique():
#     datasets[name + "_test"] = dataset_test.get_subset(dataset_test.df['dataset'] == name)
# datasets

In [36]:
# dataset_train.plot_grid()
# analysis.display_statistics(dataset_train.df)
# dataset_train.summary
# for idx, (image, label) in enumerate(dataset_train):
#     print(image)
#     print()
#     print(label)
#     if idx == 10:
#         break

img, label = dataset_train[0]
print(img.shape)
test_img, test_label = dataset_test[1]
print(test_img.shape)

torch.Size([3, 384, 384])
torch.Size([3, 384, 384])


In [ ]:
import timm

model_name = "hf-hub:BVRA/MegaDescriptor-L-384"
backbone = timm.create_model(model_name, num_classes=0, pretrained=True)
import numpy as np
from wildlife_tools.features import DeepFeatures
from wildlife_tools.similarity import CosineSimilarity
from wildlife_tools.inference import KnnClassifier


extractor = DeepFeatures(backbone, batch_size=4, device='cuda')

idx_train = list(range(10)) + list(range(190,200))
idx_test = list(range(10,20)) + list(range(200,210))
dataset_database = dataset_train.get_subset(list(range(0, 100)))
dataset_query = dataset_train.get_subset(list(range(200, 300)))
query, database = extractor(dataset_query), extractor(dataset_database)
similarity_function = CosineSimilarity()
similarity = similarity_function(query, database)

classifier = KnnClassifier(k=1, database_labels=dataset_database.labels_string)
predictions = classifier(similarity)
accuracy = np.mean(dataset_query.labels_string == predictions)

100%|███████████████████████████████████████████████████████████████| 25/25 [00:05<00:00,  4.17it/s]


In [52]:
# print(dataset_query.labels_string)
# print()
# print(predictions)
# print(accuracy)
print(query[0][0].shape)
print(database[0][0].shape)

(1536,)
(1536,)
